# U2-03 · *Bootstrap* y validación cruzada

*Machine Learning e Inteligencia Artificial (IFCD0174) · Unidad 2, apartado 2.9*

Reproduce el ejemplo 2.7 y muestra el compromiso entre sesgo y varianza con datos simulados.

Ejecuta cada celda con **Mayús + Intro**. Puedes cambiar los datos y volver a ejecutar.

## 1. *Bootstrap* de la mediana (Efron, 1979)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

tiempos = np.array([41.2, 39.8, 40.5, 42.1, 40.9, 39.5, 41.7, 40.3, 51.0, 40.1])
rng = np.random.default_rng()          # sin semilla: cada ejecución da un resultado algo distinto
B = 10_000
medianas = np.array([np.median(rng.choice(tiempos, size=len(tiempos), replace=True)) for _ in range(B)])
print(f'Mediana {np.median(tiempos):.2f} s · error típico bootstrap {medianas.std(ddof=1):.2f} s')
print('IC 95 % (percentiles):', np.percentile(medianas, [2.5, 97.5]))

In [ ]:
plt.hist(medianas, bins=30, color='#1C8C96'); plt.xlabel('Mediana de cada remuestra (s)'); plt.ylabel('Frecuencia'); plt.show()

## 2. Sesgo y varianza: ajustar polinomios de grado creciente

Se simulan 25 puntos de una curva con ruido y se ajustan polinomios de grado 1 a 10.

In [ ]:
rng = np.random.default_rng(0)
x = np.sort(rng.uniform(0, 1, 25))
y = np.sin(2 * np.pi * x) + rng.normal(0, 0.3, x.size)

def ecm(grado, x_ent, y_ent, x_pru, y_pru):
    coef = np.polyfit(x_ent, y_ent, grado)
    return np.mean((np.polyval(coef, x_pru) - y_pru) ** 2)

## 3. Validación cruzada de *k* = 5 particiones (Stone, 1974)

In [ ]:
def validacion_cruzada(grado, x, y, k=5, semilla=1):
    idx = np.random.default_rng(semilla).permutation(x.size)
    bloques = np.array_split(idx, k)
    errores = []
    for b in bloques:
        ent = np.setdiff1d(idx, b)
        errores.append(ecm(grado, x[ent], y[ent], x[b], y[b]))
    return np.mean(errores)

grados = range(1, 11)
err_ent = [ecm(g, x, y, x, y) for g in grados]
err_cv = [validacion_cruzada(g, x, y) for g in grados]
for g, e1, e2 in zip(grados, err_ent, err_cv):
    print(f'grado {g:2d}   entrenamiento {e1:.3f}   validación cruzada {e2:.3f}')

In [ ]:
plt.plot(grados, err_ent, 'o-', color='grey', label='entrenamiento')
plt.plot(grados, err_cv, 'o-', color='#1C8C96', label='validación cruzada')
plt.xlabel('Grado del polinomio (complejidad)'); plt.ylabel('Error cuadrático medio'); plt.yscale('log'); plt.legend(); plt.show()
print('Grado con menor error de validación cruzada:', grados[int(np.argmin(err_cv))])

El error de entrenamiento no deja de bajar; el de validación cruzada baja y luego sube. Compáralo con la figura 2.9 del libro.

> Efron, B. (1979). Bootstrap Methods. *The Annals of Statistics*, 7(1), 1–26. https://doi.org/10.1214/aos/1176344552  
> Stone, M. (1974). Cross-Validatory Choice and Assessment of Statistical Predictions. *JRSS B*, 36(2), 111–147. https://doi.org/10.1111/j.2517-6161.1974.tb00994.x